In [1]:
import os
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle API token configured successfully.")

Kaggle API token configured successfully.


In [2]:
!kaggle datasets download -d adityajn105/flickr8k

!unzip -q flickr8k.zip -d flickr8k_dataset

print("Dataset downloaded and extracted to the 'flickr8k_dataset' folder.")

Dataset URL: https://www.kaggle.com/datasets/adityajn105/flickr8k
License(s): CC0-1.0
100% 1.04G/1.04G [00:26<00:00, 42.0MB/s]

Dataset downloaded and extracted to the 'flickr8k_dataset' folder.


In [3]:
import pandas as pd

dataset_path = 'flickr8k_dataset'
images_folder = os.path.join(dataset_path, 'Images')
captions_file = os.path.join(dataset_path, 'captions.txt')

df = pd.read_csv(captions_file)

print(f"Total caption pairs found: {len(df)}")
print("\n--- Sample Data ---")
print(df.head())

unique_images = df['image'].unique()
print(f"\nTotal unique images: {len(unique_images)}")

val_image_names = unique_images[-1000:]
train_image_names = unique_images[:-1000]

train_df = df[df['image'].isin(train_image_names)].reset_index(drop=True)
val_df = df[df['image'].isin(val_image_names)].reset_index(drop=True)

print(f"\nTraining set: {len(train_df)} captions (from {len(train_image_names)} images)")
print(f"Validation set: {len(val_df)} captions (from {len(val_image_names)} images)")

Total caption pairs found: 40455

--- Sample Data ---
                       image  \
0  1000268201_693b08cb0e.jpg   
1  1000268201_693b08cb0e.jpg   
2  1000268201_693b08cb0e.jpg   
3  1000268201_693b08cb0e.jpg   
4  1000268201_693b08cb0e.jpg   

                                             caption  
0  A child in a pink dress is climbing up a set o...  
1              A girl going into a wooden building .  
2   A little girl climbing into a wooden playhouse .  
3  A little girl climbing the stairs to her playh...  
4  A little girl in a pink dress going into a woo...  

Total unique images: 8091

Training set: 35455 captions (from 7091 images)
Validation set: 5000 captions (from 1000 images)


In [4]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from collections import Counter

class Vocabulary:
    def __init__(self, freq_threshold):
        self.freq_threshold = freq_threshold

        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.idx = 4

    def __len__(self):
        return len(self.itos)

    def tokenize(self, text):
        return nltk.tokenize.word_tokenize(text.lower())

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()

        for sentence in sentence_list:
            for word in self.tokenize(sentence):
                frequencies[word] += 1

        for word, count in frequencies.items():
            if count >= self.freq_threshold:
                self.stoi[word] = self.idx
                self.itos[self.idx] = word
                self.idx += 1

    def numericalize(self, text):
        tokenized_text = self.tokenize(text)
        return [self.stoi.get(word, self.stoi["<UNK>"]) for word in tokenized_text]

print("Vocabulary class defined successfully.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Vocabulary class defined successfully.


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [5]:
from PIL import Image
import torch
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

class FlickrDataset(Dataset):
    def __init__(self, root_dir, df, transform=None, vocab=None, freq_threshold=5):
        self.root_dir = root_dir
        self.df = df
        self.transform = transform

        self.imgs = self.df["image"]
        self.captions = self.df["caption"]

        if vocab is None:
            self.vocab = Vocabulary(freq_threshold)
            self.vocab.build_vocabulary(self.captions.tolist())
        else:
            self.vocab = vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        caption = self.captions[index]
        img_id = self.imgs[index]

        img_path = os.path.join(self.root_dir, img_id)
        img = Image.open(img_path).convert("RGB") # Ensure 3 color channels

        if self.transform is not None:
            img = self.transform(img)

        numericalized_caption = [self.vocab.stoi["<SOS>"]]
        numericalized_caption += self.vocab.numericalize(caption)
        numericalized_caption.append(self.vocab.stoi["<EOS>"])

        return img, torch.tensor(numericalized_caption)

class CapsCollate:
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        imgs = [item[0].unsqueeze(0) for item in batch]
        imgs = torch.cat(imgs, dim=0)

        targets = [item[1] for item in batch]

        targets = pad_sequence(targets, batch_first=True, padding_value=self.pad_idx)

        return imgs, targets

print("Dataset and Collate functions defined successfully.")

Dataset and Collate functions defined successfully.


In [6]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

dataset_path = 'flickr8k_dataset'
images_folder = os.path.join(dataset_path, 'Images')

train_dataset = FlickrDataset(
    root_dir=images_folder,
    df=train_df,
    transform=transform
)

val_dataset = FlickrDataset(
    root_dir=images_folder,
    df=val_df,
    transform=transform,
    vocab=train_dataset.vocab
)

batch_size = 64
pad_idx = train_dataset.vocab.stoi["<PAD>"]

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    num_workers=2,
    shuffle=True,
    collate_fn=CapsCollate(pad_idx=pad_idx)
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=2,
    shuffle=False,
    collate_fn=CapsCollate(pad_idx=pad_idx)
)

print(f"Total vocabulary size: {len(train_dataset.vocab)}")
print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")

Total vocabulary size: 2794
Number of training batches: 554
Number of validation batches: 79


In [7]:
import torch
import torch.nn as nn
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using hardware: {device}")

class EncoderCNN(nn.Module):
    def __init__(self, embed_size):
        super(EncoderCNN, self).__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

        for param in resnet.parameters():
            param.requires_grad = False

        self.resnet = resnet
        self.resnet.fc = nn.Linear(resnet.fc.in_features, embed_size)
        self.bn = nn.BatchNorm1d(embed_size, momentum=0.01)

    def forward(self, images):
        features = self.resnet(images)
        features = self.bn(features)
        return features

class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=1):
        super(DecoderRNN, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, features, captions):
        embeddings = self.embed(captions[:, :-1])
        features = features.unsqueeze(1)
        inputs = torch.cat((features, embeddings), dim=1)
        lstm_out, _ = self.lstm(inputs)
        outputs = self.linear(lstm_out)
        return outputs

Using hardware: cuda


In [8]:
import torch.optim as optim
import torch.nn as nn

embed_size = 256
hidden_size = 512
vocab_size = len(train_dataset.vocab)
num_layers = 1
learning_rate = 3e-4
num_epochs = 5

encoder = EncoderCNN(embed_size).to(device)
decoder = DecoderRNN(embed_size, hidden_size, vocab_size, num_layers).to(device)

pad_idx = train_dataset.vocab.stoi["<PAD>"]
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

params = list(decoder.parameters()) + list(encoder.resnet.fc.parameters())
optimizer = optim.Adam(params, lr=learning_rate)

print(f"Models initialized and moved to {device}.")
print(f"Vocabulary Size: {vocab_size}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 118MB/s]


Models initialized and moved to cuda.
Vocabulary Size: 2794


In [9]:
from tqdm import tqdm

print("Starting Training Engine...")

for epoch in range(num_epochs):
    print(f"\n=== Epoch {epoch+1}/{num_epochs} ===")

    # ==========================
    #       TRAINING PHASE
    # ==========================
    encoder.train()
    decoder.train()
    train_loss = 0.0

    # Wrap train_loader in tqdm for a progress bar
    loop = tqdm(train_loader, total=len(train_loader), desc="Training")

    for idx, (imgs, captions) in enumerate(loop):
        # 1. Move tensors to GPU
        imgs = imgs.to(device)
        captions = captions.to(device)

        # 2. Zero the gradients
        optimizer.zero_grad()

        # 3. Forward pass
        features = encoder(imgs)
        outputs = decoder(features, captions)

        # 4. Calculate loss
        # Outputs: (batch_size, sequence_length, vocab_size) -> Reshape to (batch_size * sequence_length, vocab_size)
        # Captions: (batch_size, sequence_length) -> Reshape to (batch_size * sequence_length)
        loss = criterion(outputs.view(-1, vocab_size), captions.view(-1))

        # 5. Backward pass
        loss.backward()

        # 6. Clip gradients to prevent exploding gradients in the LSTM
        torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)

        # 7. Update weights
        optimizer.step()

        # Update progress bar
        train_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)

    # ==========================
    #      VALIDATION PHASE
    # ==========================
    encoder.eval()
    decoder.eval()
    val_loss = 0.0

    # Disable gradient tracking during validation to save memory and speed up computation
    with torch.no_grad():
        for imgs, captions in val_loader:
            imgs = imgs.to(device)
            captions = captions.to(device)

            features = encoder(imgs)
            outputs = decoder(features, captions)

            loss = criterion(outputs.view(-1, vocab_size), captions.view(-1))
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1} Completed | Train Loss: {avg_train_loss:.4f} | Validation Loss: {avg_val_loss:.4f}")

# ==========================
#      MODEL EXPORT
# ==========================
print("\nTraining Complete. Exporting Model Weights...")
torch.save(encoder.state_dict(), 'encoder_weights.pth')
torch.save(decoder.state_dict(), 'decoder_weights.pth')
print("✅ Weights saved successfully as .pth files!")

Starting Training Engine...

=== Epoch 1/5 ===


Training: 100%|██████████| 554/554 [03:42<00:00,  2.49it/s, loss=3.04]


Epoch 1 Completed | Train Loss: 3.7578 | Validation Loss: 3.0542

=== Epoch 2/5 ===


Training: 100%|██████████| 554/554 [03:40<00:00,  2.51it/s, loss=2.72]


Epoch 2 Completed | Train Loss: 2.9317 | Validation Loss: 2.7689

=== Epoch 3/5 ===


...

Epoch 3 Completed | Train Loss: 2.6667 | Validation Loss: 2.6162

=== Epoch 4/5 ===


Training: 100%|██████████| 554/554 [03:40<00:00,  2.51it/s, loss=2.36]


Epoch 4 Completed | Train Loss: 2.4870 | Validation Loss: 2.5257

=== Epoch 5/5 ===


Training: 100%|██████████| 554/554 [03:41<00:00,  2.51it/s, loss=2.41]


Epoch 5 Completed | Train Loss: 2.3524 | Validation Loss: 2.4649

Training Complete. Exporting Model Weights...
✅ Weights saved successfully as .pth files!
